This is task 2 of Pandahat Adverserial Learning Path. Why are me and my partner doing this? Well, me and my partner needed to test the difference between a Support Vector Machine (SVM) and a Neural Network. Specifically, their difference in performance. With the data of task 1, we trained these models to see which one was more accurate. 

# Support Vector Machine


Before training the SVM, we need to do some data preprocessing. This could be done by just using the samples from task 1 and iterrating through them, but it wouldn't have matched with the results from task 1, as the training data wouldn't be nearly as diversed as that would imply giving a general NDVI value to each image. That would throw the data out of proportion as it would lose it's diversity. So I decided to grab a random number of pixels from each image and use that for training. 

It goes as follows:

Imports and configuration for SVM:

In [41]:
from PIL import Image
import numpy as np
import os
import glob
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

class_to_id = {
    "Water/Cloud": 0,
    "Bare Soil": 1,
    "Sparse Vegetation": 2,
    "Dense Vegetation": 3
}
print(os.getcwd())
print("labels exists:", os.path.exists(labels_path))
print("samples exists:", os.path.exists(samples_path))

/Users/revelvelazquez/Pandahat Adverserial/Learning-Path/Task_2
labels exists: False
samples exists: False


Loading directories:

In [42]:
base_dir = os.path.join(os.getcwd(), "..")
labels_path = os.path.join(base_dir, "labels")
samples_path = os.path.join(base_dir, "samples")

sample_files = sorted(glob.glob(os.path.join(samples_path, "*.tiff")))
label_files = sorted(glob.glob(os.path.join(labels_path, "*.tiff")))

print(f"Found {len(sample_files)} samples")
print(f"Found {len(label_files)} labels")

Found 614 samples
Found 614 labels


Building the dataset:

In [43]:
dataset = []

for i in range(len(sample_files)):
    img = Image.open(sample_files[i])
    label = Image.open(label_files[i])

    img_array = np.array(img)
    label_array = np.array(label)
    ndvi_array = (label_array / 255.0) * 2 - 1

    height = img_array.shape[0]
    width = img_array.shape[1]

    # Chose 50 pixels for speed, tried 500 but it simply took way too long. 
    # Waited about 58 minutes and it didn't finish testing. This is mostly fault of the
    # RBF kernel but I stuck with it as I thought it would be the best choice
    # for training data that is extremely non-linear.
    rows = np.random.randint(0, height, size=50)
    cols = np.random.randint(0, width, size=50)

    for r, c in zip(rows, cols):
        red   = img_array[r, c, 0]
        green = img_array[r, c, 1]
        blue  = img_array[r, c, 2]
        ndvi_val = ndvi_array[r, c]

        if ndvi_val < 0:
            veg_class = "Water/Cloud"
        elif ndvi_val < 0.2:
            veg_class = "Bare Soil"
        elif ndvi_val < 0.5:
            veg_class = "Sparse Vegetation"
        else:
            veg_class = "Dense Vegetation"

        dataset.append([red, green, blue, class_to_id[veg_class]])

print(f"Total pixels collected: {len(dataset)}")

Total pixels collected: 30700


Converting, scaling and splitting the dataset to a numpy array:

*Note scaling, although not mandatory here, it's best practice to do so.

In [44]:
dataset_array = np.array(dataset)
X = dataset_array[:, 0:3]
y = dataset_array[:, 3]

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  
X_test_scaled  = scaler.transform(X_test)   


Features shape: (30700, 3)
Labels shape: (30700,)
Training samples: 24560
Test samples: 6140


Training (this takes some time):

In [45]:
print("\nTraining SVM...")
svm = SVC(kernel='rbf', random_state=42, class_weight='balanced')
svm.fit(X_train_scaled, y_train)
print("Training complete!")


Training SVM...
Training complete!


Evaluation:

In [46]:
print("\nEvaluating...")
y_pred = svm.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Water/Cloud', 'Bare Soil', 'Sparse Veg', 'Dense Veg']
))


Evaluating...

Test Accuracy: 0.7370

Classification Report:
              precision    recall  f1-score   support

 Water/Cloud       0.32      0.65      0.42       333
   Bare Soil       0.67      0.54      0.60      1170
  Sparse Veg       0.77      0.74      0.75      2436
   Dense Veg       0.87      0.85      0.86      2201

    accuracy                           0.74      6140
   macro avg       0.66      0.69      0.66      6140
weighted avg       0.76      0.74      0.74      6140



RESULTS: 

Found 614 samples
Found 614 labels
Total pixels collected: 30700
Features shape: (30700, 3)
Labels shape: (30700,)
Training samples: 24560
Test samples: 6140

Training SVM...
Training complete!

Evaluating...

Test Accuracy: 0.7479

Classification Report:
              precision    recall  f1-score   support

 Water/Cloud       0.34      0.67      0.45       327
   Bare Soil       0.67      0.56      0.61      1203
  Sparse Veg       0.78      0.73      0.76      2428
   Dense Veg       0.86      0.88      0.87      2182

    accuracy                           0.75      6140
   macro avg       0.67      0.71      0.67      6140
weighted avg       0.77      0.75      0.75      6140


Explaining the data:

Overall accuracy: 74%

Water/Clouds: when it came to this class, it was 34% accurate. Why is that? It's mainly because within the data there aren't enough water or clouds pixels, it's mostly guessing on this classification. This matches up pretty well on task 1's data as there wasn't a lot of water or clouds on the samples, so it fails on this class. This is supported by the support column: there's only 327 pixels. On the recall it found 67%, but those are mostly false alarms. The F1-score summarizes the precision and the recall, 45%.

Bare Soil: 67% accurate, according to the support column, it had 1203 pixels to learn from, much better than the water/clouds class. On recall it caught 56% pixels, although less than water/clouds, it likely had to guess a lot less, explaining the lower percentage. F1-score summary of precision and recall, 61%.

Spares Vegetation: 78% acuracy, much better. This is due to the abundance of vegetation pixels. On recall it had caught 73% of the pixels. It had 2428 pixels to train itself, checks out with the informtion from task 1. The F1-score summarizes the precision and the recall, 76%.

Dense vegetation: 86% accuracy, best performance overall. On the data from task 1 there was mostly dense vegetation. It had less pixels than the sparse vegetation class according to the support class, 2182 pixels. On recall it caught 88% of the pixels. he F1-score summarizes the precision and the recall, 87%.

Accuracy: Overall, 75% of pixels were classified correctly.

Macro avg: Average across classes without considering class size; shows Water/Cloud hurts the average.

Weighted avg: Average considering the number of samples per class; dominated by Dense Veg and Sparse Veg, hence higher.
